<a href="https://colab.research.google.com/github/aisha13dikko-sudo/using-synthetic-data-for-thermal-comfort-classification/blob/main/wk4_mostlyai_autotherm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U mostlyai --quiet
import mostlyai
print("✅ MostlyAI installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.8/243.8 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 97.5 MB/s eta 0:00:00
✅ MostlyAI installed


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from mostlyai.sdk import MostlyAI

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import re

print("✅ Imports ready")

✅ Imports ready


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
!pip install datasets --quiet

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [5]:
print("Loading AutoTherm indoor dataset...")
dataset = load_dataset('kopetri/AutoTherm', 'indoor')
train_df = dataset['train'].to_pandas()

def extract_participant_id(filename):
    match = re.search(r'participant_\d+', filename)
    return match.group() if match else 'unknown'

train_df['participant_id'] = train_df['file_name'].apply(extract_participant_id)

def add_3class(df):
    df = df.copy()
    df['Label_3class'] = df['Label'].apply(
        lambda x: -1 if x <= -2 else (0 if x <= 1 else 1)
    )
    return df

train_df = add_3class(train_df)

test_participants  = ['participant_16', 'participant_14', 'participant_20']
train_participants = [p for p in train_df['participant_id'].unique()
                      if p not in test_participants]

train_split = train_df[train_df['participant_id'].isin(train_participants)]
test_split  = train_df[train_df['participant_id'].isin(test_participants)]

print(f"Train: {len(train_split):,} | Test: {len(test_split):,}")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Loading AutoTherm indoor dataset...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

indoor/train-00000-of-00002.parquet:   0%|          | 0.00/29.8M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Generating test split:   0%|          | 0/194829 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Train: 1,276,709 | Test: 290,019


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [6]:
drop_cols = [
    'file_name', 'Timestamp', 'participant_id',
    'Air-Velocity', 'Metabolic-Rate',
    'Nose', 'Neck', 'RShoulder', 'RElbow',
    'LShoulder', 'LElbow', 'REye', 'LEye', 'REar', 'LEar',
    'Emotion-Self', 'Emotion-ML',
    'Label', 'Label_3class'
]

def prepare_features(df, drop_cols, target_col):
    df = df.copy()
    X = df.drop(columns=drop_cols, errors='ignore')
    y = df[target_col]
    for col in X.select_dtypes(include=['object','category']).columns:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    return X, y

def prepare_synthetic(df, feature_cols, target_col):
    X = df[feature_cols].copy()
    y = df[target_col]
    for col in X.select_dtypes(include=['object','category']).columns:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    X = X.fillna(X.median(numeric_only=True))
    return X, y

X_train_7, y_train_7 = prepare_features(train_split, drop_cols, 'Label')
X_test_7,  y_test_7  = prepare_features(test_split,  drop_cols, 'Label')
X_train_3, y_train_3 = prepare_features(train_split, drop_cols, 'Label_3class')
X_test_3,  y_test_3  = prepare_features(test_split,  drop_cols, 'Label_3class')

feature_cols = X_train_7.columns.tolist()
print(f"Features: {len(feature_cols)}")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Features: 18


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [7]:
clf_base_7 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_base_7.fit(X_train_7, y_train_7)
y_pred_base_7 = clf_base_7.predict(X_test_7)
base_7_f1  = f1_score(y_test_7, y_pred_base_7, average='macro')
base_7_bal = balanced_accuracy_score(y_test_7, y_pred_base_7)

clf_base_3 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_base_3.fit(X_train_3, y_train_3)
y_pred_base_3 = clf_base_3.predict(X_test_3)
base_3_f1  = f1_score(y_test_3, y_pred_base_3, average='macro')
base_3_bal = balanced_accuracy_score(y_test_3, y_pred_base_3)

print(f"Baseline 7-class: {base_7_f1:.4f} | 3-class: {base_3_f1:.4f}")

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replac

Baseline 7-class: 0.2858 | 3-class: 0.7163


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [8]:
sdv_drop = [
    'file_name', 'Timestamp', 'participant_id',
    'Air-Velocity', 'Metabolic-Rate',
    'Nose', 'Neck', 'RShoulder', 'RElbow',
    'LShoulder', 'LElbow', 'REye', 'LEye', 'REar', 'LEar',
    'Emotion-Self', 'Emotion-ML',
    'Label_3class'
]

mostly_train = train_split.drop(columns=sdv_drop, errors='ignore').copy()
mostly_train['Gender'] = LabelEncoder().fit_transform(mostly_train['Gender'].astype(str))

# Same 100K stratified sample for fair comparison with CTGAN/TVAE
mostly_train_sample = mostly_train.groupby('Label', group_keys=False).apply(
    lambda x: x.sample(
        n=min(len(x), int(100000 * len(x) / len(mostly_train))),
        random_state=42
    )
).reset_index(drop=True)

print(f"Sample size: {len(mostly_train_sample):,}")
print(mostly_train_sample['Label'].value_counts().sort_index())

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Sample size: 99,996
Label
-3     3761
-2     9151
-1    20622
 0    24056
 1    13968
 2    15896
 3    12542
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/tmp/ipykernel_542/2756754299.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mostly_train_sample = mostly_train.groupby('Label', group_keys=False).apply(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datet

In [9]:
!pip install -U "mostlyai[local]" --quiet 2>&1 | tail -20
print("✅ Installed mostlyai[local]")

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replac

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.8/287.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.7/54.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.9/208.9 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 132.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ Installed mostlyai[local]


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [10]:
mostly = MostlyAI(local=True)

print("Training MostlyAI generator on AutoTherm sample...")
g = mostly.train(
    name='autotherm_indoor',
    data=mostly_train_sample
)
print("\n✅ MostlyAI generator trained!")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Initializing Synthetic Data SDK 6.1.1 in LOCAL mode 🏠

Connected to ]8;id=492148;file:///root/mostlyai\/root/]8;;\]8;id=631771;file:///root/mostlyai\mostlyai]8;;\ with 83 GB RAM, 12 CPUs, 1x NVIDIA A100-SXM4-40GB available

Training MostlyAI generator on AutoTherm sample...


Created generator 515f8ffe-1d62-495d-88fe-02eb001a0847

Started generator training

Output()

🎉 Your generator is ready! Use it to create synthetic data. Publish it so others can do the same.


✅ MostlyAI generator trained!


In [11]:
print("Generating synthetic data from trained generator...")
mostly_synthetic = mostly.probe(g, size=len(mostly_train_sample))
print(f"✅ Generated {len(mostly_synthetic):,} synthetic rows")

print("\nColumns in synthetic data:")
print(mostly_synthetic.columns.tolist())

print("\nLabel distribution — REAL (sample):")
print(mostly_train_sample['Label'].value_counts().sort_index())
print("\nLabel distribution — MOSTLYAI SYNTHETIC:")
print(mostly_synthetic['Label'].value_counts().sort_index())

print("\nFirst 3 rows of synthetic data:")
print(mostly_synthetic.head(3))

Generating synthetic data from trained generator...
✅ Generated 99,996 synthetic rows

Columns in synthetic data:
['Age', 'Gender', 'Weight', 'Height', 'Bodyfat', 'Bodytemp', 'Sport-Last-Hour', 'Time-Since-Meal', 'Tiredness', 'Clothing-Level', 'Radiation-Temp', 'PCE-Ambient-Temp', 'Wrist_Skin_Temperature', 'Heart_Rate', 'GSR', 'Ambient_Temperature', 'Ambient_Humidity', 'Solar_Radiation', 'Label']

Label distribution — REAL (sample):
Label
-3     3761
-2     9151
-1    20622
 0    24056
 1    13968
 2    15896
 3    12542
Name: count, dtype: int64

Label distribution — MOSTLYAI SYNTHETIC:
Label
-3     3395
-2     8846
-1    21156
0     26880
1     14644
2     15552
3      9523
Name: count, dtype: Int64

First 3 rows of synthetic data:
   Age  Gender  Weight  Height  Bodyfat  Bodytemp  Sport-Last-Hour  \
0   25       0    54.2   155.0      0.0      36.5                0   
1   23       1    90.6   175.0    0.296      36.5                0   
2   26       0    59.8   176.0    0.217      3

In [12]:
from mostlyai import qa

print("Generating MostlyAI QA report...")
report_path, metrics = qa.report(
    syn_tgt_data=mostly_synthetic,
    trn_tgt_data=mostly_train_sample
)

print("\n✅ QA Report generated!")
print(metrics.model_dump_json(indent=2))

Generating MostlyAI QA report...

✅ QA Report generated!
{
  "accuracy": {
    "overall": 0.9039957,
    "univariate": 0.9559511,
    "bivariate": 0.9070001,
    "trivariate": 0.8490361,
    "coherence": null,
    "overall_max": 0.9883618,
    "univariate_max": 0.9955637,
    "bivariate_max": 0.9892192,
    "trivariate_max": 0.9803025,
    "coherence_max": null
  },
  "similarity": {
    "cosine_similarity_training_synthetic": 0.991504,
    "cosine_similarity_training_holdout": null,
    "discriminator_auc_training_synthetic": 0.757,
    "discriminator_auc_training_holdout": null
  },
  "distances": {
    "ims_training": 0.0,
    "ims_holdout": null,
    "ims_trn_hol": null,
    "dcr_training": 0.083,
    "dcr_holdout": null,
    "dcr_trn_hol": null,
    "dcr_share": null,
    "nndr_training": 0.160265542181,
    "nndr_holdout": null,
    "nndr_trn_hol": null
  }
}


In [13]:
X_mostly_synth, y_mostly_synth = prepare_synthetic(
    mostly_synthetic, feature_cols, 'Label'
)

clf_mostly_tstr_7 = RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=-1
)
clf_mostly_tstr_7.fit(X_mostly_synth, y_mostly_synth)
y_pred_mostly_tstr_7 = clf_mostly_tstr_7.predict(X_test_7)

mostly_tstr_7_f1  = f1_score(y_test_7, y_pred_mostly_tstr_7, average='macro')
mostly_tstr_7_bal = balanced_accuracy_score(y_test_7, y_pred_mostly_tstr_7)

print("📊 MostlyAI TSTR — 7-class")
print(f"Macro F1:          {mostly_tstr_7_f1:.4f}")
print(f"Balanced Accuracy: {mostly_tstr_7_bal:.4f}")
print(f"Baseline was:      {base_7_f1:.4f}")
print(f"Difference:        {mostly_tstr_7_f1 - base_7_f1:+.4f}")
print()
print(classification_report(y_test_7, y_pred_mostly_tstr_7))

📊 MostlyAI TSTR — 7-class
Macro F1:          0.2535
Balanced Accuracy: 0.2609
Baseline was:      0.2858
Difference:        -0.0323

              precision    recall  f1-score   support

          -3       0.00      0.00      0.00     22556
          -2       0.12      0.22      0.16     19285
          -1       0.60      0.71      0.65     85260
           0       0.11      0.13      0.12     39044
           1       0.01      0.01      0.01     16824
           2       0.36      0.38      0.37     51786
           3       0.61      0.37      0.46     55264

    accuracy                           0.38    290019
   macro avg       0.26      0.26      0.25    290019
weighted avg       0.38      0.38      0.37    290019



In [14]:
# Augmented
X_aug_mostly_7 = pd.concat(
    [X_train_7, X_mostly_synth], axis=0
).reset_index(drop=True)
y_aug_mostly_7 = pd.concat([
    y_train_7.reset_index(drop=True),
    y_mostly_synth.reset_index(drop=True)
], axis=0).reset_index(drop=True)

clf_mostly_aug_7 = RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=-1
)
clf_mostly_aug_7.fit(X_aug_mostly_7, y_aug_mostly_7)
y_pred_mostly_aug_7 = clf_mostly_aug_7.predict(X_test_7)

mostly_aug_7_f1  = f1_score(y_test_7, y_pred_mostly_aug_7, average='macro')
mostly_aug_7_bal = balanced_accuracy_score(y_test_7, y_pred_mostly_aug_7)

print("📊 MostlyAI AUGMENTED — 7-class")
print(f"Macro F1:          {mostly_aug_7_f1:.4f}")
print(f"Baseline was:      {base_7_f1:.4f}")
print(f"Difference:        {mostly_aug_7_f1 - base_7_f1:+.4f}")
print(classification_report(y_test_7, y_pred_mostly_aug_7))

# 3-class
mostly_synthetic_3 = mostly_synthetic.copy()
mostly_synthetic_3['Label_3class'] = mostly_synthetic_3['Label'].apply(
    lambda x: -1 if x <= -2 else (0 if x <= 1 else 1)
)

X_mostly_synth_3, y_mostly_synth_3 = prepare_synthetic(
    mostly_synthetic_3, feature_cols, 'Label_3class'
)

clf_mostly_tstr_3 = RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=-1
)
clf_mostly_tstr_3.fit(X_mostly_synth_3, y_mostly_synth_3)
y_pred_mostly_tstr_3 = clf_mostly_tstr_3.predict(X_test_3)
mostly_tstr_3_f1 = f1_score(y_test_3, y_pred_mostly_tstr_3, average='macro')

X_aug_mostly_3 = pd.concat(
    [X_train_3, X_mostly_synth_3], axis=0
).reset_index(drop=True)
y_aug_mostly_3 = pd.concat([
    y_train_3.reset_index(drop=True),
    y_mostly_synth_3.reset_index(drop=True)
], axis=0).reset_index(drop=True)

clf_mostly_aug_3 = RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=-1
)
clf_mostly_aug_3.fit(X_aug_mostly_3, y_aug_mostly_3)
y_pred_mostly_aug_3 = clf_mostly_aug_3.predict(X_test_3)
mostly_aug_3_f1 = f1_score(y_test_3, y_pred_mostly_aug_3, average='macro')

print(f"\n📊 MostlyAI TSTR 3-class:      {mostly_tstr_3_f1:.4f} | Baseline: {base_3_f1:.4f}")
print(f"📊 MostlyAI Augmented 3-class: {mostly_aug_3_f1:.4f} | Baseline: {base_3_f1:.4f}")

📊 MostlyAI AUGMENTED — 7-class
Macro F1:          0.2682
Baseline was:      0.2858
Difference:        -0.0176
              precision    recall  f1-score   support

          -3       0.00      0.00      0.00     22556
          -2       0.21      0.56      0.30     19285
          -1       0.67      0.62      0.64     85260
           0       0.09      0.08      0.08     39044
           1       0.05      0.12      0.07     16824
           2       0.34      0.39      0.36     51786
           3       0.61      0.31      0.41     55264

    accuracy                           0.36    290019
   macro avg       0.28      0.30      0.27    290019
weighted avg       0.40      0.36      0.37    290019


📊 MostlyAI TSTR 3-class:      0.7003 | Baseline: 0.7163
📊 MostlyAI Augmented 3-class: 0.7066 | Baseline: 0.7163


In [15]:
# Complete results with MostlyAI included
results_with_mostly = pd.DataFrame({
    'Method': [
        'Baseline', 'GC TSTR', 'GC Augmented',
        'CTGAN TSTR', 'CTGAN Augmented',
        'TVAE TSTR', 'TVAE Augmented',
        'SMOTE', 'MostlyAI TSTR', 'MostlyAI Augmented',
    ],
    '7-class Macro F1': [
        0.2858, 0.2727, 0.2825, 0.1709, 0.1841,
        0.2438, 0.2677, 0.2586,
        mostly_tstr_7_f1, mostly_aug_7_f1
    ],
    '3-class Macro F1': [
        0.7163, 0.4969, 0.4932, 0.5040, 0.4925,
        0.6802, 0.6460, 0.6517,
        mostly_tstr_3_f1, mostly_aug_3_f1
    ],
}).round(4)

print(results_with_mostly.to_string(index=False))
results_with_mostly.to_csv('wk4_complete_results_with_mostlyai.csv', index=False)

            Method  7-class Macro F1  3-class Macro F1
          Baseline            0.2858            0.7163
           GC TSTR            0.2727            0.4969
      GC Augmented            0.2825            0.4932
        CTGAN TSTR            0.1709            0.5040
   CTGAN Augmented            0.1841            0.4925
         TVAE TSTR            0.2438            0.6802
    TVAE Augmented            0.2677            0.6460
             SMOTE            0.2586            0.6517
     MostlyAI TSTR            0.2535            0.7003
MostlyAI Augmented            0.2682            0.7066


In [16]:
# ── CLASS-WEIGHTED RANDOM FOREST ──────────────────────────────────
# Same real training data, same real test data — NO synthetic data,
# NO resampling. We only change how the classifier WEIGHTS each
# class during training. This avoids the prior-shift collapse we
# saw with full/partial oversampling.

print("Training class-weighted Random Forest — 7-class...")

clf_weighted_7 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'   # <-- the only change from baseline
)
clf_weighted_7.fit(X_train_7, y_train_7)
y_pred_weighted_7 = clf_weighted_7.predict(X_test_7)

weighted_7_f1  = f1_score(y_test_7, y_pred_weighted_7, average='macro')
weighted_7_bal = balanced_accuracy_score(y_test_7, y_pred_weighted_7)

print(f"\n📊 CLASS-WEIGHTED RF — 7-class")
print(f"Macro F1:          {weighted_7_f1:.4f}")
print(f"Balanced Accuracy: {weighted_7_bal:.4f}")
print(f"Baseline was:      {base_7_f1:.4f}")
print(f"Difference:        {weighted_7_f1 - base_7_f1:+.4f}")
print()
print(classification_report(y_test_7, y_pred_weighted_7))

Training class-weighted Random Forest — 7-class...

📊 CLASS-WEIGHTED RF — 7-class
Macro F1:          0.2504
Balanced Accuracy: 0.2626
Baseline was:      0.2858
Difference:        -0.0354

              precision    recall  f1-score   support

          -3       0.00      0.00      0.00     22556
          -2       0.03      0.03      0.03     19285
          -1       0.61      0.71      0.66     85260
           0       0.10      0.15      0.12     39044
           1       0.15      0.32      0.20     16824
           2       0.36      0.33      0.35     51786
           3       0.60      0.30      0.40     55264

    accuracy                           0.37    290019
   macro avg       0.26      0.26      0.25    290019
weighted avg       0.38      0.37      0.36    290019



In [17]:
print("Training class-weighted Random Forest — 3-class...")

clf_weighted_3 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
clf_weighted_3.fit(X_train_3, y_train_3)
y_pred_weighted_3 = clf_weighted_3.predict(X_test_3)

weighted_3_f1  = f1_score(y_test_3, y_pred_weighted_3, average='macro')
weighted_3_bal = balanced_accuracy_score(y_test_3, y_pred_weighted_3)

print(f"\n📊 CLASS-WEIGHTED RF — 3-class")
print(f"Macro F1:          {weighted_3_f1:.4f}")
print(f"Balanced Accuracy: {weighted_3_bal:.4f}")
print(f"Baseline was:      {base_3_f1:.4f}")
print(f"Difference:        {weighted_3_f1 - base_3_f1:+.4f}")
print()
print(classification_report(y_test_3, y_pred_weighted_3))

Training class-weighted Random Forest — 3-class...

📊 CLASS-WEIGHTED RF — 3-class
Macro F1:          0.6815
Balanced Accuracy: 0.6667
Baseline was:      0.7163
Difference:        -0.0348

              precision    recall  f1-score   support

          -1       0.70      0.62      0.66     41841
           0       0.64      0.90      0.75    141128
           1       0.93      0.49      0.64    107050

    accuracy                           0.71    290019
   macro avg       0.76      0.67      0.68    290019
weighted avg       0.76      0.71      0.69    290019



In [18]:
!pip install imbalanced-learn --quiet

from imblearn.ensemble import BalancedRandomForestClassifier

print("Training BalancedRandomForestClassifier — 7-class...")

clf_brf_7 = BalancedRandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    sampling_strategy='all',
    replacement=True
)
clf_brf_7.fit(X_train_7, y_train_7)
y_pred_brf_7 = clf_brf_7.predict(X_test_7)

brf_7_f1  = f1_score(y_test_7, y_pred_brf_7, average='macro')
brf_7_bal = balanced_accuracy_score(y_test_7, y_pred_brf_7)

print(f"\n📊 BALANCED RANDOM FOREST — 7-class")
print(f"Macro F1:          {brf_7_f1:.4f}")
print(f"Baseline was:      {base_7_f1:.4f}")
print(f"Difference:        {brf_7_f1 - base_7_f1:+.4f}")
print()
print(classification_report(y_test_7, y_pred_brf_7))

Training BalancedRandomForestClassifier — 7-class...

📊 BALANCED RANDOM FOREST — 7-class
Macro F1:          0.2429
Baseline was:      0.2858
Difference:        -0.0429

              precision    recall  f1-score   support

          -3       0.00      0.00      0.00     22556
          -2       0.06      0.08      0.07     19285
          -1       0.62      0.70      0.66     85260
           0       0.05      0.09      0.07     39044
           1       0.15      0.31      0.20     16824
           2       0.39      0.36      0.38     51786
           3       0.63      0.22      0.32     55264

    accuracy                           0.35    290019
   macro avg       0.27      0.25      0.24    290019
weighted avg       0.39      0.35      0.35    290019



In [19]:
# ── OVERLAP DIAGNOSTIC: Is -3 actually separable from -2/-1? ──────
# If even a simple classifier struggles to tell Cold apart from
# its neighbours using ONLY real data, that's strong evidence the
# classes genuinely overlap in feature space — no generator can
# manufacture a boundary that doesn't exist.

from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

# Isolate just the real training rows for labels -3 and -2
# (the two most easily confused neighbours)
mask_32 = y_train_7.isin([-3, -2])
X_32 = X_train_7[mask_32]
y_32 = y_train_7[mask_32]

print(f"Rows with label -3: {(y_32 == -3).sum():,}")
print(f"Rows with label -2: {(y_32 == -2).sum():,}")

# Simple binary classifier: can it tell -3 apart from -2 at all?
clf_binary = RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced'
)
scores = cross_val_score(clf_binary, X_32, y_32, cv=5, scoring='f1_macro')

print(f"\n5-fold CV Macro F1 separating -3 vs -2 (real data only):")
print(f"Scores: {scores.round(4)}")
print(f"Mean:   {scores.mean():.4f}")
print(f"\nFor reference: 0.50 = no separation (random guessing)")
print(f"               1.00 = perfect separation")

Rows with label -3: 48,020
Rows with label -2: 116,840

5-fold CV Macro F1 separating -3 vs -2 (real data only):
Scores: [0.6587 0.4845 0.359  1.     1.    ]
Mean:   0.7005

For reference: 0.50 = no separation (random guessing)
               1.00 = perfect separation


In [20]:
# Same idea but binary: is -3 separable from ALL other classes combined?
y_binary = (y_train_7 == -3).astype(int)

clf_vs_all = RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced'
)
scores_vs_all = cross_val_score(
    clf_vs_all, X_train_7, y_binary, cv=5, scoring='f1_macro'
)

print(f"5-fold CV Macro F1 separating -3 vs ALL other classes:")
print(f"Scores: {scores_vs_all.round(4)}")
print(f"Mean:   {scores_vs_all.mean():.4f}")

5-fold CV Macro F1 separating -3 vs ALL other classes:
Scores: [0.8077 0.6161 0.4919 0.9533 0.7095]
Mean:   0.7157


In [21]:
# ── DOES SEPARABILITY DEPEND ON WHICH PARTICIPANT IS HELD OUT? ────
# Leave-one-participant-out, but only for participants who have -3

participants_with_cold = train_df[train_df['Label'] == -3]['participant_id'].unique()
print(f"Participants with label -3 in training pool: {sorted(participants_with_cold)}")

from sklearn.metrics import f1_score as f1s

for held_out in sorted(participants_with_cold):
    train_subset = train_df[
        (train_df['participant_id'] != held_out) &
        (train_df['Label'].isin([-3, -2]))
    ]
    test_subset = train_df[
        (train_df['participant_id'] == held_out) &
        (train_df['Label'].isin([-3, -2]))
    ]

    if len(test_subset) == 0 or test_subset['Label'].nunique() < 2:
        print(f"{held_out}: skipped (insufficient -3/-2 rows)")
        continue

    X_tr, y_tr = prepare_features(train_subset, drop_cols, 'Label')
    X_te, y_te = prepare_features(test_subset, drop_cols, 'Label')

    clf_lopo = RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced'
    )
    clf_lopo.fit(X_tr, y_tr)
    y_pred_lopo = clf_lopo.predict(X_te)

    f1_lopo = f1s(y_te, y_pred_lopo, average='macro')
    print(f"{held_out}: held-out -3 vs -2 F1 = {f1_lopo:.4f}  (n={len(test_subset)})")

Participants with label -3 in training pool: ['participant_11', 'participant_16', 'participant_19', 'participant_2']
participant_11: held-out -3 vs -2 F1 = 0.4176  (n=35812)
participant_16: held-out -3 vs -2 F1 = 0.1883  (n=29367)
participant_19: held-out -3 vs -2 F1 = 0.3500  (n=35501)
participant_2: held-out -3 vs -2 F1 = 0.2006  (n=28711)


In [22]:
# ── IS PARTICIPANT_16'S COLD DATA "INSIDE" THE OTHER PARTICIPANTS' RANGE? ──
# If their cold readings fall within the min/max range the other 12
# participants ALREADY showed for each feature, a generator trained on
# those 12 has a chance of interpolating into that region.
# If their readings sit OUTSIDE that range, no generator can reach it —
# that would be true extrapolation, which generators cannot do reliably.

other_participants_data = train_df[
    (train_df['participant_id'] != 'participant_16') &
    (train_df['participant_id'].isin(train_participants))
]
p16_cold_data = train_df[
    (train_df['participant_id'] == 'participant_16') &
    (train_df['Label'] == -3)
]

print(f"participant_16 cold rows: {len(p16_cold_data)}")
print(f"\n{'Feature':<28} {'Other ppts range':<28} {'p16 cold range':<28} {'Inside?'}")
print("-" * 95)

key_features = ['Ambient_Temperature', 'Wrist_Skin_Temperature', 'Radiation-Temp',
                 'PCE-Ambient-Temp', 'Ambient_Humidity', 'GSR']

for feat in key_features:
    other_min, other_max = other_participants_data[feat].min(), other_participants_data[feat].max()
    p16_min, p16_max = p16_cold_data[feat].min(), p16_cold_data[feat].max()
    inside = "YES" if (p16_min >= other_min and p16_max <= other_max) else "NO — partial/outside"
    print(f"{feat:<28} [{other_min:.2f}, {other_max:.2f}]{'':<8} [{p16_min:.2f}, {p16_max:.2f}]{'':<8} {inside}")

participant_16 cold rows: 22556

Feature                      Other ppts range             p16 cold range               Inside?
-----------------------------------------------------------------------------------------------
Ambient_Temperature          [17.60, 37.00]         [18.50, 23.30]         YES
Wrist_Skin_Temperature       [27.91, 36.95]         [32.25, 34.43]         YES
Radiation-Temp               [16.90, 33.60]         [17.60, 23.20]         YES
PCE-Ambient-Temp             [17.10, 33.70]         [17.80, 23.10]         YES
Ambient_Humidity             [12.00, 55.00]         [40.00, 50.00]         YES
GSR                          [0.00, 6.47]         [0.01, 0.35]         YES
